This notebook is to demonstrate and analyze the search results of the Retrieval-Augmented Generation (RAG) system.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
from mistralai.models import SDKError

c:\Users\gsjsc\anaconda3\envs\en605645\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
module_path = os.path.abspath(os.path.join('..'))
sys.path.append(module_path)
from pipeline import Pipeline
from generator.question_answering import QA_Generator

In [3]:
# Define path for sample test
corpus_path = "..\\storage\\corpus"
faiss_path = "..\\storage\\index\\faiss_index_corpus.bin"
metadata_path = "..\\storage\\index\\metadata_corpus.pkl"

In [13]:
# Initiate pipeline on corpus
pipeline = Pipeline(index_type='brute_force', rerank_type="hybrid", temperature=0.8, generator_model="mistral-large-latest")

In [5]:
# Preprocess the document and save the index
pipeline.preprocess_corpus(corpus_path, chunking_strategy='sentence', fixed_length=None, overlap_size=2)
pipeline.indexer.save(faiss_path, metadata_path)

In [14]:
# Load the precomputed index
pipeline.load_index(faiss_path, metadata_path)

# 1. Demonstrate output from search_neighbors() function

In [6]:
# Define input queries and k
queries = [
    "Who was Abraham Lincoln?",
    "Who was Abraham Adams?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "How did Fillmore ascend to the presidency?",
    "What is the capital of France?",
]

k_values = [15,15,1,10,20,50,1,3]

In [7]:
for query, k in zip(queries, k_values):
    pipeline.search_neighbors(query, k)
    print("____________________________________________________________________\n")

QUERY: Who was Abraham Lincoln?

NEAREST NEIGHBORS RESULTS:
Neighbor 1: Index 454, Distance 0.7003927230834961, Documents: Abraham Lincoln Abraham Lincoln (February 12, 1809 – April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would describe his protest in the Illinois legislature as one that 'briefly defined his position on the slavery question, and so far as it goes, it was then the same that it is now."
Neighbor 2: Index 565, Distance 0.7469339370727539, Documents: Lincoln is well known for ending slavery in the United States. In 1861 – 1862, however, he made it clear that the North was fighting the war to preserve the Union, not to abolish slavery.
Neighbor 3: Index 663, Distance 0.7481499910354614, Documents: Lincoln, Illinois, is the only city to be named for 

## Observations on search_neighbors()

- **High relevance for known-entities.** For similar queries "Who was Abraham Lincoln?" and "Who was Abraham Adams?" and same k for both, more relevant documents are returned for "Abraham Lincoln" due to this well-know figure. For "Who was Abraham Adams?", the model retrieved documents mainly related to historical Adams family figures like John Adams and John Quincy Adams. This can also be supported by the average distances of the retrived documents for the two queries.

- **Noise in Higher k Results.** When increasing k from 1 to 10, 20, or 50, for "Did Abraham Lincoln live in the Frontier?", more documents introduce tangential information, leading to less focus on the primary query topic. This suggests a trade-off where increasing k can increase recall but may reduce precision and teke more memory and time, introducing unrelated information into the response pool.

- **Out-of-Scope Query Handling.** For queries such as "What is the capital of France?", the retrieved results are contextually related to France but do not provide direct answers about France's capital. This might due to limitations of the corpus.



# 2. Demonstrate output from generate_answer() function

In [4]:
# Define input queries, k
queries = [
    "Who was Abraham Lincoln?",
    "Who was Abraham Adams?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "Did Abraham Lincoln live in the Frontier?",
    "What trail did Lincoln use a Farmers' Almanac in?",
    "What trail did Lincoln use a Farmers' Almanac in?",
    "What trail did Lincoln use a Farmers' Almanac in?",
    "What trail did Lincoln use a Farmers' Almanac in?",
    "What is the capital of France?",
]

k_values = [15,15,1,5,10,20,1,5,10,20,15]


## 2.1 Test on low temeprature to provide more deterministic and focused outputs

In [5]:
# Test on low temeprature to provide more deterministic and focused outputs
pipeline = Pipeline(index_type='brute_force', rerank_type="hybrid", temperature=0.2, generator_model="mistral-large-latest")
# Load the precomputed index
pipeline.load_index(faiss_path, metadata_path)

In [6]:
for query, k in zip(queries, k_values):
    for rerank in [True, False]:
        print(f"For top ({k}) retrived documents and when rerank = {rerank}:\n")
        retrived_docs = pipeline.search_neighbors(query, k, reporting=False)
        pipeline.generate_answer(query, retrived_docs, rerank=rerank)
        print("____________________________________________________________________\n")

For top (15) retrived documents and when rerank = True:

[ChatCompletionChoice(index=0, message=AssistantMessage(content='Abraham Lincoln was the sixteenth President of the United States, serving from March 4, 1861 until his assassination on April 15, 1865.', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
QUERY: Who was Abraham Lincoln?

GENERATED ANSWER: Abraham Lincoln was the sixteenth President of the United States, serving from March 4, 1861 until his assassination on April 15, 1865.
____________________________________________________________________

For top (15) retrived documents and when rerank = False:

[ChatCompletionChoice(index=0, message=AssistantMessage(content='Abraham Lincoln was the sixteenth President of the United States, serving from March 4, 1861 until his assassination on April 15, 1865.', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
QUERY: Who was Abraham Lincoln?

GENERATED ANSWER: Abraham Lincoln was the s

## 2.2 Test on high temeprature to provide more creative and diverse outputs

In [7]:
# Test on high temeprature to provide more creative and diverse outputs
pipeline = Pipeline(index_type='brute_force', rerank_type="hybrid", temperature=0.8, generator_model="mistral-large-latest")
# Load the precomputed index
pipeline.load_index(faiss_path, metadata_path)

In [8]:
for query, k in zip(queries, k_values):
    for rerank in [True, False]:
        print(f"For top ({k}) retrived documents and when rerank = {rerank}:\n")
        retrived_docs = pipeline.search_neighbors(query, k, reporting=False)
        pipeline.generate_answer(query, retrived_docs, rerank=rerank)
        print("____________________________________________________________________\n")

For top (15) retrived documents and when rerank = True:

[ChatCompletionChoice(index=0, message=AssistantMessage(content='Abraham Lincoln was the sixteenth President of the United States, serving from March 4, 1861 until his assassination on April 15, 1865.', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
QUERY: Who was Abraham Lincoln?

GENERATED ANSWER: Abraham Lincoln was the sixteenth President of the United States, serving from March 4, 1861 until his assassination on April 15, 1865.
____________________________________________________________________

For top (15) retrived documents and when rerank = False:

[ChatCompletionChoice(index=0, message=AssistantMessage(content='Abraham Lincoln (February 12, 1809 – April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination.', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
QUERY: Who was Abraham Lincoln?

GENERATED ANSWER: Abraha

## Observations on generate_answer()

- **Deterministic Responses at Low Temperature.** Answers at temperature 0.2 are consistent, factual, and straightforward. This is useful for tasks where predictability and accuracy are key. However, the answers are lack of variation since the model provide limited diversity. The model returns "No context" a lot when it does not find relevant information. This possibly indicates over-reliance on the retrieved documents.

- **Increased Variability at High Temperature.** The model starts to return varied wording for “No context” responses and more elaborate responses. Especially when reranking is disabled, the model is more likely to provide answers that attempt to utilize partial matches in the documents.

- **Effect of Reranking.** Reranking helps in refining answers and therefore improves the relevance in answers. With reranking enabled, the model is more precise in filtering irrelevant information, especially when multiple documents are retrieved. This is evident in the “No context” responses when reranking is on, as it prevents partial matches from confusing the model. Without reranking, higher temperature and k values result in more verbose answers. This increased the diversity of responses but also introduces some degree of speculation.
